In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression

# --- CONFIGURAZIONE PERCORSI ---
# Adatta il path se il file ha un nome diverso
PATH_INPUT = 'data_elaborated/train/train_cleaned.csv' 
DIR_OUTPUT = 'data_elaborated/train/'
FILE_OUTPUT = 'train_with_physics_residuals.csv'

# Numero di cicli considerati "motore sano" (Baseline per i residui)
HEALTHY_CYCLES_THRESHOLD = 30

In [2]:
def add_physics_features(df):
    print("-> 1. Applicazione formule fisiche (Ciclo Brayton - Unità: Rankine/PSI)...")
    df = df.copy()
    
    # Costanti Standard a Livello del Mare (SLS)
    STD_TEMP_RANKINE = 518.67     # Gradi Rankine 
    STD_PRES_PSI = 14.696         # PSI (1 atm)
    GAMMA_AIR = 1.4
    
    # Check colonne base necessarie
    required = ['Sensed_T25', 'Sensed_Pt2', 'Sensed_T45', 'Sensed_Ps3', 'Sensed_T3']
    if not all(col in df.columns for col in required):
        print("ERR: Mancano sensori fondamentali per la fisica.")
        return df

    # --- A. Calcolo Theta e Delta (Nessuna somma +273.15 perché siamo in Rankine!) ---
    theta = df['Sensed_T25'] / STD_TEMP_RANKINE 
    delta = df['Sensed_Pt2'] / STD_PRES_PSI
    
    # Protezione matematica contro le divisioni per zero
    theta = theta.replace(0, 1)
    delta = delta.replace(0, 1)

    # --- B. Parametri Corretti (Rimozione effetto altitudine/Mach) ---
    df['Phy_T45_Corr'] = df['Sensed_T45'] / theta
    
    if 'Sensed_WFuel' in df.columns:
        df['Phy_WFuel_Corr'] = df['Sensed_WFuel'] / (delta * np.sqrt(theta))
        
    if 'Sensed_Core_Speed' in df.columns:
        df['Phy_Core_Speed_Corr'] = df['Sensed_Core_Speed'] / np.sqrt(theta)

    # --- C. Efficienza Isentropica (Compressore di Alta Pressione - HPC) ---
    T_in_R = df['Sensed_T25']
    T_out_R = df['Sensed_T3']
    
    # Usa Sensed_P25 se esiste, altrimenti usa Pt2 come fallback
    P_in = df['Sensed_P25'] if 'Sensed_P25' in df.columns else df['Sensed_Pt2']
    P_out = df['Sensed_Ps3']
    
    pr = P_out / P_in
    k = (GAMMA_AIR - 1) / GAMMA_AIR
    
    # Temperatura ideale (in Rankine) ed efficienza
    T_iso_R = T_in_R * (pr ** k)
    df['Phy_Compressor_Eff'] = (T_iso_R - T_in_R) / (T_out_R - T_in_R)
    
    # Heat Rate
    df['Phy_Heat_Index'] = df['Sensed_T45'] / df['Sensed_Ps3']

    return df

In [3]:
def calculate_physics_residuals(df, healthy_threshold):
    print(f"-> 2. Calcolo Residui (Addestramento baseline sui primi {healthy_threshold} cicli)...")
    
    # Usiamo solo le variabili esterne per modellare gli effetti aerodinamici residui
    X_cols = ['Sensed_Altitude', 'Sensed_Mach']
    if 'Sensed_Pamb' in df.columns: X_cols.append('Sensed_Pamb')
    X_cols = [c for c in X_cols if c in df.columns]
    
    y_cols = [
        'Phy_T45_Corr', 'Phy_WFuel_Corr', 
        'Phy_Core_Speed_Corr', 'Phy_Compressor_Eff', 'Phy_Heat_Index'
    ]
    y_cols = [c for c in y_cols if c in df.columns]
    
    engine_results = []
    motori = df['ESN'].unique()
    
    for esn in motori:
        engine_df = df[df['ESN'] == esn].copy()
        engine_df = engine_df.sort_values('Cycles_Since_New')
        
        # Pulizia da eventuali NaN
        engine_df = engine_df.dropna(subset=y_cols + X_cols)
        if engine_df.empty: continue
        
        for target in y_cols:
            X_all = engine_df[X_cols].values
            y_all = engine_df[target].values
            
            # --- LA CORREZIONE CRITICA: Filtriamo i dati SANI ---
            healthy_mask = engine_df['Cycles_Since_New'] <= healthy_threshold
            X_healthy = engine_df.loc[healthy_mask, X_cols].values
            y_healthy = engine_df.loc[healthy_mask, target].values
            
            # Fallback di sicurezza se un motore non ha registrato i primi cicli
            if len(X_healthy) < 10:
                n_fallback = max(int(len(X_all) * 0.1), 5) # Prendi il primo 10%
                X_healthy = X_all[:n_fallback]
                y_healthy = y_all[:n_fallback]
            
            # Regressione addestrata SOLO sul motore in stato ottimale
            model = LinearRegression()
            model.fit(X_healthy, y_healthy)
            
            # Previsione e Calcolo Residuo su TUTTA LA VITA
            y_pred_all = model.predict(X_all)
            engine_df[f"{target}_res"] = y_all - y_pred_all
            
        engine_results.append(engine_df)
        
    return pd.concat(engine_results, ignore_index=True)

In [4]:
# --- AVVIO PROCESSO ---

# Fallback in caso di percorso errato (cerca nella stessa cartella se non trova il file)
if not os.path.exists(PATH_INPUT) and os.path.exists('training_data_cleaned.csv'):
    PATH_INPUT = 'training_data_cleaned.csv'

if os.path.exists(PATH_INPUT):
    print(f"Caricamento file originale: {PATH_INPUT}...")
    df_raw = pd.read_csv(PATH_INPUT)
    
    # Step 1: Normalizzazione Fisica
    df_phys = add_physics_features(df_raw)
    
    # Step 2: Calcolo Degradazione (Residui Fisici)
    df_final = calculate_physics_residuals(df_phys, HEALTHY_CYCLES_THRESHOLD)
    
    # Step 3: Salvataggio
    if not os.path.exists(DIR_OUTPUT): 
        os.makedirs(DIR_OUTPUT)
        
    out_path = os.path.join(DIR_OUTPUT, FILE_OUTPUT)
    df_final.to_csv(out_path, index=False)
    
    print("-" * 50)
    print("✅ SUCCESSO!")
    print(f"Nuovo Dataset creato e salvato in: {out_path}")
    print("\nEcco le nuove Features Fisiche e Residui che puoi usare nel Machine Learning:")
    
    nuove_colonne = [c for c in df_final.columns if 'Phy_' in c]
    for col in nuove_colonne:
        print(f" - {col}")

else:
    print(f"❌ ERRORE: Il file {PATH_INPUT} non è stato trovato.")
    print("Assicurati di trovarti nella cartella corretta o modifica il path in Cella 1.")

Caricamento file originale: data_elaborated/train/train_cleaned.csv...
-> 1. Applicazione formule fisiche (Ciclo Brayton - Unità: Rankine/PSI)...
-> 2. Calcolo Residui (Addestramento baseline sui primi 30 cicli)...
--------------------------------------------------
✅ SUCCESSO!
Nuovo Dataset creato e salvato in: data_elaborated/train/train_with_physics_residuals.csv

Ecco le nuove Features Fisiche e Residui che puoi usare nel Machine Learning:
 - Phy_T45_Corr
 - Phy_WFuel_Corr
 - Phy_Core_Speed_Corr
 - Phy_Compressor_Eff
 - Phy_Heat_Index
 - Phy_T45_Corr_res
 - Phy_WFuel_Corr_res
 - Phy_Core_Speed_Corr_res
 - Phy_Compressor_Eff_res
 - Phy_Heat_Index_res


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer 
import re


def generate_submission_file(folder_path, models_dict, feature_sets, output_file='risultati/submission.csv'):
    search_path = os.path.join(folder_path, "*.csv")
    all_files = glob.glob(search_path)
    
    # --- 1. ORDINAMENTO NUMERICO DEI FILE ---
    def extract_number(filepath):
        filename = os.path.basename(filepath)
        match = re.search(r'\d+', filename)
        return int(match.group()) if match else 0

    all_files_sorted = sorted(all_files, key=extract_number)

    results = []
    print(f"\n--- GENERAZIONE SUBMISSION FILE ({len(all_files_sorted)} files) ---")
    
    for file_path in all_files_sorted:
        filename = os.path.basename(file_path)
        file_id = filename.replace('.csv', '') 
        
        try: df_test_raw = pd.read_csv(file_path)
        except: continue
        if df_test_raw.empty: continue
        
        # Preprocessing (Fisica al volo per il test set)
        df_test_raw = harmonize_columns(df_test_raw)
        df_mech = prepare_mechanical_data(df_test_raw, is_test=True) 
        df_wash = prepare_wash_data(df_test_raw, is_test=True)       
        
        data_map = {'HPT': df_mech, 'HPC': df_mech, 'WW': df_wash}
        row_pred = {'file': file_id} 
        
        for ctype in ['HPT', 'HPC', 'WW']:
            model = models_dict[ctype]['model']
            margin = models_dict[ctype]['margin']
            imputer = models_dict[ctype]['imputer'] # Usiamo l'imputer addestrato!
            df = data_map[ctype]
            features = feature_sets[ctype]
            
            # Se il df è vuoto (es. filtri troppo restrittivi), mettiamo 0 di default
            if df.empty:
                row_pred[ctype] = 0
                continue
            
            for f in features: 
                if f not in df.columns: df[f] = 0
            
            X_test = df[features]
            X_test = X_test.replace([np.inf, -np.inf], np.nan)
            
            try:
                # Usiamo transform, NON fit_transform!
                X_clean = pd.DataFrame(imputer.transform(X_test), columns=features)
                X_clean = X_clean.fillna(0) 
                
                pred_raw = model.predict(X_clean)
                pred_safe = np.maximum(pred_raw - margin, 0)
                final_val = pred_safe[-1] if len(pred_safe) > 0 else 0
                row_pred[ctype] = final_val
            except Exception as e: 
                print(f"Errore su {filename} ({ctype}): {e}")
                row_pred[ctype] = 0
                
        results.append(row_pred)
        
    # --- 3. FORMATTAZIONE ESATTA PER LA COMPETIZIONE ---
    df_submission = pd.DataFrame(results)
    
    df_submission = df_submission.rename(columns={
        'WW': 'Cycles_to_WW',
        'HPC': 'Cycles_to_HPC_SV',
        'HPT': 'Cycles_to_HPT_SV'
    })
    
    for col in ['Cycles_to_WW', 'Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']:
        df_submission[col] = df_submission[col].round(0).astype(int)
        
    df_submission = df_submission[['file', 'Cycles_to_WW', 'Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']]
    
    # Crea cartella risultati se non esiste
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df_submission.to_csv(output_file, index=False)
    print(f"File salvato e pronto per l'upload: {output_file}")


# ==============================================================================
# 0. MOTORE FISICO (Corretto in Rankine!)
# ==============================================================================
def harmonize_columns(df):
    rename_map = {
        'Cycles': 'Cycles_Since_New', 'Cycle': 'Cycles_Since_New',
        'Altitude': 'Sensed_Altitude', 'Mach': 'Sensed_Mach',
        'TRA': 'Sensed_TRA', 'T2': 'Sensed_T2', 'T24': 'Sensed_T24',
        'T30': 'Sensed_T3', 'T48': 'Sensed_T45', 'T50': 'Sensed_T5',
        'P15': 'Sensed_P15', 'P2': 'Sensed_P2', 'P21': 'Sensed_P21',
        'P24': 'Sensed_P24', 'Ps30': 'Sensed_Ps3', 'P40': 'Sensed_P40',
        'P50': 'Sensed_P50', 'HPC_SV': 'Cycles_to_HPC_SV',
        'HPT_SV': 'Cycles_to_HPT_SV', 'WW': 'Cycles_to_WW'
    }
    df = df.rename(columns=rename_map)
    if 'Sensed_Altitude' not in df.columns and 'Altitude' not in df.columns:
         for col in df.columns:
             if col not in ['ESN', 'Cycles_Since_New', 'Snapshot', 'File_ID']:
                 if not col.startswith('Sensed_') and not col.startswith('Phy_'):
                     df.rename(columns={col: f'Sensed_{col}'}, inplace=True)
    return df

def add_physics_features(df):
    df = df.copy()
    STD_TEMP_R = 518.67; STD_PRES = 14.696; GAMMA_AIR = 1.4

    if 'Sensed_T25' in df.columns and 'Sensed_Pt2' in df.columns:
        theta = df['Sensed_T25'] / STD_TEMP_R 
        delta = df['Sensed_Pt2'] / STD_PRES
        theta = theta.replace(0, 1); delta = delta.replace(0, 1)

        if 'Sensed_Core_Speed' in df.columns:
            df['Phy_Core_Speed_Corr'] = df['Sensed_Core_Speed'] / np.sqrt(theta)
        if 'Sensed_WFuel' in df.columns:
            df['Phy_WFuel_Corr'] = df['Sensed_WFuel'] / (delta * np.sqrt(theta))
        if 'Sensed_T45' in df.columns:
            df['Phy_T45_Corr'] = df['Sensed_T45'] / theta

        if 'Sensed_T3' in df.columns and 'Sensed_Ps3' in df.columns:
            T_in_R = df['Sensed_T25'] 
            T_out_R = df['Sensed_T3'] 
            P_in = df['Sensed_P25'] if 'Sensed_P25' in df.columns else df['Sensed_Pt2']
            P_out = df['Sensed_Ps3']
            
            pr = P_out / P_in
            k = (GAMMA_AIR - 1) / GAMMA_AIR
            T_iso_R = T_in_R * (pr ** k)
            df['Phy_Compressor_Eff'] = (T_iso_R - T_in_R) / (T_out_R - T_in_R)

    if 'Sensed_T45' in df.columns and 'Sensed_Ps3' in df.columns:
        df['Phy_Heat_Index'] = df['Sensed_T45'] / df['Sensed_Ps3']

    return df

# ==============================================================================
# 1. PREPROCESSING
# ==============================================================================
def prepare_mechanical_data(df, is_test=False):
    df = df.copy()
    df = harmonize_columns(df)
    
    if 'Sensed_Altitude' in df.columns:
        df = df[df['Sensed_Altitude'] > 20000].copy()
    
    df = add_physics_features(df)

    phy_cols = ['Phy_T45_Corr', 'Phy_Compressor_Eff', 'Phy_Heat_Index', 'Phy_Core_Speed_Corr']
    res_cols = [c + '_res' for c in phy_cols]
    raw_cols = ['Sensed_Ps3', 'Sensed_T3'] 
    
    use_cols = [c for c in phy_cols + res_cols + raw_cols if c in df.columns]
    
    agg_dict = {col: 'mean' for col in use_cols}
    targets = ['Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']
    for t in targets:
        if t in df.columns: agg_dict[t] = 'first'
        
    if not is_test and 'ESN' in df.columns:
        df_grouped = df.groupby(['ESN', 'Cycles_Since_New']).agg(agg_dict).reset_index()
        df_grouped = df_grouped.sort_values(['ESN', 'Cycles_Since_New'])
        for col in use_cols:
            df_grouped[f"{col}_smooth"] = df_grouped.groupby('ESN')[col].transform(
                lambda x: x.rolling(window=10, min_periods=1).mean()
            )
    else:
        df_grouped = df.groupby('Cycles_Since_New').agg(agg_dict).reset_index()
        df_grouped = df_grouped.sort_values('Cycles_Since_New')
        for col in use_cols:
            df_grouped[f"{col}_smooth"] = df_grouped[col].rolling(window=10, min_periods=1).mean()

    df_grouped = df_grouped.ffill().bfill().fillna(0)
    return df_grouped

def prepare_wash_data(df, is_test=False):
    df = df.copy()
    df = harmonize_columns(df)
    
    if 'Sensed_Core_Speed' in df.columns:
        df = df[df['Sensed_Core_Speed'] > 8000].copy()

    df = add_physics_features(df)
    
    phy_cols = ['Phy_Compressor_Eff', 'Phy_Heat_Index', 'Phy_WFuel_Corr']
    res_cols = [c + '_res' for c in phy_cols]
    raw_backup = ['Sensed_WFuel', 'Sensed_T45'] 
    
    use_cols = [c for c in phy_cols + res_cols + raw_backup if c in df.columns]
    
    agg_dict = {}
    for c in use_cols: agg_dict[c] = ['mean', 'max']
    
    if 'Cycles_to_WW' in df.columns: agg_dict['Cycles_to_WW'] = 'first'
    if 'Cumulative_WWs' in df.columns: agg_dict['Cumulative_WWs'] = 'max' 
    
    if not is_test and 'ESN' in df.columns:
        df_grouped = df.groupby(['ESN', 'Cycles_Since_New']).agg(agg_dict)
    else:
        df_grouped = df.groupby('Cycles_Since_New').agg(agg_dict)
        
    new_cols = []
    feature_cols = []
    for c, s in df_grouped.columns:
        if c in ['Cycles_to_WW', 'Cumulative_WWs'] or s == '': new_cols.append(c)
        else: 
            name = f"{c}_{s}"
            new_cols.append(name)
            feature_cols.append(name)
    
    df_grouped.columns = new_cols
    df_grouped = df_grouped.reset_index()
    
    if not is_test and 'ESN' in df.columns:
        df_grouped = df_grouped.sort_values(['ESN', 'Cycles_Since_New'])
        if 'Cumulative_WWs' in df_grouped.columns:
             df_grouped['WW_Change'] = df_grouped.groupby('ESN')['Cumulative_WWs'].diff().fillna(0)
             df_grouped['Wash_Session_ID'] = df_grouped.groupby('ESN')['WW_Change'].cumsum()
             df_grouped['Cycles_Since_Last_Wash'] = df_grouped.groupby(['ESN', 'Wash_Session_ID']).cumcount()
             feature_cols.append('Cycles_Since_Last_Wash')

        for col in feature_cols:
            if col == 'Cycles_Since_Last_Wash': continue 
            for i in range(1, 4): 
                df_grouped[f"{col}_lag{i}"] = df_grouped.groupby('ESN')[col].shift(i)
            df_grouped[f"{col}_smooth"] = df_grouped.groupby('ESN')[col].transform(
                lambda x: x.rolling(window=5, min_periods=1).mean()
            )
            df_grouped[f"{col}_diff"] = df_grouped.groupby('ESN')[col].diff()
    else:
        df_grouped = df_grouped.sort_values('Cycles_Since_New')
        if 'Cumulative_WWs' in df_grouped.columns:
             df_grouped['WW_Change'] = df_grouped['Cumulative_WWs'].diff().fillna(0)
             df_grouped['Wash_Session_ID'] = df_grouped['WW_Change'].cumsum()
             df_grouped['Cycles_Since_Last_Wash'] = df_grouped.groupby('Wash_Session_ID').cumcount()
             feature_cols.append('Cycles_Since_Last_Wash')
             
        for col in feature_cols:
            if col == 'Cycles_Since_Last_Wash': continue 
            for i in range(1, 4): 
                df_grouped[f"{col}_lag{i}"] = df_grouped[col].shift(i)
            df_grouped[f"{col}_smooth"] = df_grouped[col].rolling(window=5, min_periods=1).mean()
            df_grouped[f"{col}_diff"] = df_grouped[col].diff()

    df_grouped = df_grouped.ffill().bfill().fillna(0)
    drop_cols = ['WW_Change', 'Wash_Session_ID', 'Cumulative_WWs']
    df_grouped = df_grouped.drop(columns=[c for c in drop_cols if c in df_grouped.columns])
    
    return df_grouped

# ==============================================================================
# 2. METRICHE & UTILS (REINSERITE!)
# ==============================================================================
def get_score(y_true, y_pred, component_type):
    error = y_pred - y_true
    alpha = 0.01
    max_val = np.max(y_true) if len(y_true) > 0 else 1
    beta = 1/max_val if component_type == 'WW' else 2/max_val
    w = np.where(error >= 0, 2/(1+alpha*y_true), 1/(1+alpha*y_true))
    return np.mean(w * (error**2) * beta)

def get_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# ==============================================================================
# 3. MAIN
# ==============================================================================
def main():
    print("--- CARICAMENTO TRAINING (FISICA + RESIDUI) ---")
    try:
        df_raw = pd.read_csv('data_elaborated/train/train_with_physics_residuals.csv')
    except:
        print("File non trovato."); return
    
    df_raw = harmonize_columns(df_raw)

    print("\n--- DATASET PREPARATION ---")
    df_mech = prepare_mechanical_data(df_raw, is_test=False)
    df_wash = prepare_wash_data(df_raw, is_test=False)
    
    tasks = [
        {'target': 'Cycles_to_HPT_SV', 'data': df_mech, 'type': 'HPT', 'alpha': 0.05, 'margin': 300}, 
        {'target': 'Cycles_to_HPC_SV', 'data': df_mech, 'type': 'HPC', 'alpha': 0.05, 'margin': 300},
        {'target': 'Cycles_to_WW',     'data': df_wash, 'type': 'WW',  'alpha': 0.10, 'margin': 15}  
    ]

    trained_models = {}     
    feature_sets = {}

    print("\n============================================================")
    print(" AVVIO TRAINING FINALE E VERIFICA PERFORMANCE")
    print("============================================================")

    for task in tasks:
        target = task['target']
        df = task['data']
        ctype = task['type']
        alpha_q = task['alpha']
        margin = task['margin']
        
        if target not in df.columns: continue
        
        features = [c for c in df.columns if c not in ['ESN', target, 'Snapshot'] and 'Cycles_to_' not in c]
        feature_sets[ctype] = features 
        
        X = df[features]
        y = df[target]
        
        # Addestriamo l'imputer
        imputer = SimpleImputer(strategy='mean')
        X_clean = pd.DataFrame(imputer.fit_transform(X), columns=features)
        
        if ctype == 'WW':
            params = {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'alpha': alpha_q, 'subsample': 1.0}
        else:
            params = {'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.05, 'alpha': alpha_q, 'subsample': 0.5}
            
        print(f">>> Modello {ctype} (Feats: {len(features)})")
        model = GradientBoostingRegressor(
            loss='quantile', alpha=params['alpha'], 
            n_estimators=params['n_estimators'], 
            learning_rate=params['learning_rate'], 
            max_depth=params['max_depth'], 
            subsample=params['subsample'],
            random_state=42
        )
        # Addestramento
        model.fit(X_clean, y)
        
        # --- CALCOLO METRICHE ---
        y_pred_raw = model.predict(X_clean)
        y_pred_safe = np.maximum(y_pred_raw - margin, 0) # Applichiamo lo stesso margine usato nel test
        
        rmse_val = get_rmse(y, y_pred_safe)
        score_val = get_score(y, y_pred_safe, ctype)
        
        print(f"    [TRAIN RESULT] RMSE: {rmse_val:.2f} | Score: {score_val:.4f}\n")
        
        # Salvataggio nel dizionario
        trained_models[ctype] = {'model': model, 'margin': margin, 'target_name': target, 'imputer': imputer}

    TEST_FOLDER_PATH = 'data/val/'  
    if os.path.exists(TEST_FOLDER_PATH):
        generate_submission_file(TEST_FOLDER_PATH, trained_models, feature_sets, output_file='risultati/submission_val.csv')

    REAL_TEST_PATH = 'data/test/' 
    if os.path.exists(REAL_TEST_PATH):
        print(f"\n--- AVVIO GENERAZIONE SUBMISSION FINALE (Test Set) ---")
        generate_submission_file(REAL_TEST_PATH, trained_models, feature_sets, output_file='risultati/submission_final_test.csv')

if __name__ == "__main__":
    main()

--- CARICAMENTO TRAINING (FISICA + RESIDUI) ---

--- DATASET PREPARATION ---

 AVVIO TRAINING FINALE E VERIFICA PERFORMANCE
>>> Modello HPT (Feats: 21)
    [TRAIN RESULT] RMSE: 992.35 | Score: 19.4177

>>> Modello HPC (Feats: 21)
    [TRAIN RESULT] RMSE: 1821.62 | Score: 14.0344

>>> Modello WW (Feats: 98)
    [TRAIN RESULT] RMSE: 98.50 | Score: 1.3897


--- GENERAZIONE SUBMISSION FILE (48 files) ---
File salvato e pronto per l'upload: risultati/submission_val.csv

--- AVVIO GENERAZIONE SUBMISSION FINALE (Test Set) ---

--- GENERAZIONE SUBMISSION FILE (52 files) ---


/Users/niccolo/Desktop/progetto phm america/A - PHM America 2025/venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/niccolo/Desktop/progetto phm america/A - PHM America 2025/venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/niccolo/Desktop/progetto phm america/A - PHM America 2025/venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


File salvato e pronto per l'upload: risultati/submission_final_test.csv


## SCORE -> 183.6